## Model Context Protocol (MCP) 

is a standard framework by Anthropic that allows AI models to securely connect with external tools and data sources. It helps models access real-time information without needing custom integrations. MCP makes system integration easier, faster, and more scalable across different applications and industries.

## MCP (Model Context Protocol) Architecture


### MCP host
The LLM is contained within the MCP host, an AI application or environment such as an AI-powered IDE. This is typically the user's interaction point, where the MCP host uses the LLM to process requests that may require external data or tools.

### MCP client
The MCP client, located within the MCP host, helps the LLM and MCP server communicate with each other. It translates the LLM's requests for the MCP and converts the MCP's replies for the LLM. It also finds and uses available MCP servers.

### MCP server
The MCP server is the external service that provides context, data, or capabilities to the LLM. It helps LLMs by connecting to external systems like databases and web services, translating their responses into a format the LLM can understand which helps developers provide diverse functionalities.

### Transport layer
The transport layer uses JSON-RPC 2.0 messages to communicate between the client and server, mainly through two transport methods:

**Standard input/output (stdio)**: Works well for local resources, offering fast, synchronous message transmission              
**Server-sent events (SSE)**: Preferred for remote resources, allowing efficient, real-time data streaming

## MCP Workflow

When a user submits a request, the **LLM first analyzes the request** to determine whether external tools are required. If tools are needed, the **MCP Client discovers the appropriate tools** by communicating with the available **MCP Servers**.

The **LLM then invokes the required tool** through the MCP Client. The **MCP Server executes the requested action** by interacting with external systems such as databases, APIs, file systems, or email services, and returns the result to the LLM.

If the task requires multiple tools, the **LLM repeats the same process** until all necessary actions are completed. Finally, the **LLM combines the results and generates a natural language response**, which is sent back to the user.
        

## The Problems MCP Solves for LLms 
- Knowledge cutoffs: LLMs only know wht they know
- Hallucinations: MCP enables LLms to access more "context" from other sources
- Isolated Intelligence: LLms can't natively interact with external systems, perform actions or access private user data. 
- complex and Brittle Integrations: Before MCP, developers would need to build custom, fragile integrations for each service. 

## Advantages of MCP for LLMs and AI Agents
- Enables AI Agents: MCP allows AI agents to discover and use external tools, APIs, and data sources, enabling them to perform real-world tasks beyond text generation.
- Personalization: MCP securely provides access to user-specific data (with proper authentication and permissions), allowing AI assistants to deliver personalized responses.
- Specialized Knowledge: AI models can connect to domain-specific knowledge bases, databases, and enterprise systems to generate more accurate and expert-level responses.
- Enhanced Security: MCP centralizes authentication, authorization, and access control, ensuring secure interactions with external systems while protecting sensitive data.

## MCP Stack

### 1. Application Layer

This is the **AI application** where users interact with the LLM. It uses an MCP client to communicate with MCP servers and access external tools.

**Examples:** Claude Desktop, Cursor, VS Code with MCP, Custom AI Agents.


### 2. Protocol Layer

This layer defines the **MCP protocol**, which standardizes communication between the MCP client and MCP server. It specifies message formats, handshakes, tool discovery, resource access, and tool invocation.


### 3. Transport Layer

This layer is responsible for **how messages are transmitted** between the MCP client and MCP server. It ensures reliable communication using supported transports.

**Examples:** STDIO (local communication), HTTP, WebSocket, SSE (Server-Sent Events).


### 4. Network Layer

This is the **underlying network infrastructure** that carries messages between the client and server when they are running on different machines.

**Examples:** Local Network (LAN), Internet, VPN, Cloud Networks.


## MCP Transports

MCP Transports define how messages are exchanged between the MCP Client and the MCP Server. The transport layer is responsible only for delivering messages—it does not define the message format (the MCP Protocol does that).

### STDIO (Standard Input/Standard Output) in MCP

STDIO (Standard Input/Standard Output) is a transport mechanism where the MCP Client launches the MCP Server as a local process and communicates with it through the operating system's standard input (stdin) and standard output (stdout) streams.


### work flow 

**STDIO Workflow:**
The user sends a request to the AI application, which forwards it to the MCP Client; the MCP Client starts the local MCP Server (if needed), sends the request through **stdin** using the MCP (JSON-RPC) protocol, the MCP Server executes the requested tool or accesses the local resource, returns the result through **stdout**, and the MCP Client forwards the response to the LLM, which generates the final response for the user.


### Preferred Use Cases 

| Use Case                 | Why STDIO?                                          |
| ------------------------ | --------------------------------------------------- |
| Local file system access | Fast communication with local files.                |
| VS Code extensions       | Extension and MCP server run on the same machine.   |
| Claude Desktop           | Local MCP servers for filesystem, Git, etc.         |
| Cursor IDE               | Code assistant interacting with local repositories. |
| Development & Testing    | Easy to set up without networking.                  |
| Internal desktop tools   | No need to expose services over the network.        |

### Pro's 

STDIO provides the fastest communication because there is no network layer involved, requires no network or port configuration, offers process isolation for better security, has a simple implementation, and is ideal for local development.

### con's

STDIO works only when the client and server are on the same machine, cannot connect to remote services, tightly couples the server lifecycle to the client process, does not support load balancing, and is not suitable for web applications or multi-user deployments due to its single local server instance.

###  Server-Sent Events (SSE) 

SSE (Server-Sent Events) in MCP is an HTTP-based transport that enables the MCP Server to continuously stream real-time updates or tool execution results to the MCP Client over a single, persistent HTTP connection. It supports one-way communication, where the server sends data to the client, making it ideal for long-running tasks and streaming responses.

### Step-by-step Workflow

1. **User** asks a question in the AI application (e.g., Claude Desktop).
2. The **MCP Client** establishes a long-lived **SSE (HTTP GET)** connection with the remote MCP Server.
3. The **MCP Client** sends the user's request as a **JSON-RPC HTTP POST** request.
4. The **MCP Server** receives the request and starts executing the appropriate tool (e.g., GitHub, Database, Weather API).
5. While processing, the **MCP Server streams progress updates** over the existing SSE connection.
6. Once processing is complete, the **MCP Server streams the final result** through the same SSE connection.
7. The **MCP Client** forwards the streamed response to the AI application, which displays it to the user.

| Use Case                    | Why SSE is a Good Fit                                        |
| --------------------------- | ------------------------------------------------------------ |
| Long-running tool execution | Streams progress updates while the tool is running.          |
| RAG document indexing       | Shows indexing, embedding, and ingestion progress.           |
| Database queries            | Streams query status and partial results.                    |
| GitHub repository analysis  | Displays progress while fetching and analyzing repositories. |
| File processing             | Streams upload, parsing, and processing updates.             |
| AI/LLM response streaming   | Sends generated tokens to the client as they are produced.   |
| Monitoring dashboards       | Continuously pushes logs, metrics, or status updates.        |

### **Pros of SSE**

* **Simple, efficient, and real-time:** SSE uses standard HTTP, is lightweight, eliminates polling, and enables the server to push live updates to clients as soon as data is available.
* **Reliable and HTTP-friendly:** It supports automatic reconnection and works seamlessly with existing web infrastructure such as load balancers, proxies, firewalls, and authentication systems.

### **Cons of SSE**

* **Limited communication:** SSE supports only one-way (server-to-client) communication and transmits only UTF-8 text, requiring separate HTTP requests for client messages and encoding for binary data.
* **Scalability and modern MCP limitations:** Long-lived connections and browser connection limits can affect scalability, and the dedicated HTTP+SSE transport is now considered legacy in favor of **Streamable HTTP** in modern MCP.



### What is Streamable HTTP?

**Streamable HTTP** is the **recommended transport** in modern **Model Context Protocol (MCP)**. It allows the **client and server to communicate over a single HTTP endpoint**, while also supporting **streaming responses** when needed.

Unlike the older HTTP+SSE transport, Streamable HTTP simplifies communication by using one endpoint for requests and optionally streaming responses back to the client.


### Step-by-Step

1. The **user** sends a request in the AI application.
2. The **MCP Client** sends a **POST** request to the MCP Server.
3. The **MCP Server** starts processing the request.
4. If the task is long-running, the server **streams progress updates** over the same HTTP interaction.
5. When processing finishes, the server sends the **final JSON-RPC response**.
6. The client displays the streamed updates and final result to the user.

## Pros

* **Single HTTP endpoint:** Simpler than managing separate POST and SSE endpoints.
* **Supports streaming:** Can stream progress updates and LLM output in real time.
* **Standard HTTP:** Works well with proxies, authentication, and load balancers.
* **Flexible:** Supports both normal request/response and streaming based on the task.
* **Modern MCP standard:** Recommended transport for new MCP implementations.

## Cons

* **More complex than basic HTTP:** Streaming requires additional client and server support.
* **Long-lived connections:** Streaming requests remain open until the response completes, which can increase resource usage for many concurrent clients.

## MCP Resources

MCP Resources are read-only data sources that an MCP Server exposes to an AI application. They provide context to the LLM but do not perform any action. Think of them as files or data that the model can read whenever needed.

## MCP Tools vs MCP Resources

An MCP Resource is a read-only source of contextual information (such as files, logs, or database records), while an MCP Tool is an executable capability that performs actions and may modify external systems.


## MCP Inspector 

is an official testing and debugging tool for MCP Servers. It helps developers validate server connectivity, tools, resources, prompts, schemas, error handling, performance, and MCP protocol compliance before integrating the server with an AI client.


## In MCP, 

authentication verifies identity, authorization controls access to tools and resources based on permissions, and compliance ensures the system meets security and regulatory requirements through encryption, auditing, logging, and data protection.